# Analísis de la Isla de Calor Urbana: El Caso de Milán 🌍🔥

Este cuaderno presenta un **storytelling** basado en 6 bases de datos sobre el fenómeno de la isla de calor en Milán, analizando desde las temperaturas hasta las soluciones pasivas como techos verdes.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configuración de estilo premium
template = "plotly_dark"
colors = ['#ff4d00', '#ff9d00', '#2ecc71', '#3498db', '#9b59b6']

def load_data():
    files = {
        'consumo': 'Consumo_agua_gas_electricidad_2011.csv',
        'metros': 'Metros_cuadrados_habitante.csv',
        'verde': 'Verde_Urbano.csv',
        'temp': 'dias_calurosos_temperaturas.csv',
        'techos': 'techos_verdes_Milan.csv',
        'metano': 'zonas_sin_metano_Milan.csv'
    }
    
    dfs = {}
    for key, path in files.items():
        if 'temp' in key:
            dfs[key] = pd.read_csv(path, sep=';')
        elif 'metano' in key:
            dfs[key] = pd.read_csv(path, sep=';')
        else:
            dfs[key] = pd.read_csv(path)
    return dfs

dfs = load_data()
print("Bases de datos cargadas correctamente.")

: 

## 1. El Pulmón Verde de Milán
¿Cómo ha evolucionado el espacio verde por habitante y qué tipo de áreas predominan?

In [ ]:
def plot_green_evolution():
    df = dfs['verde']
    # Limpiar nombres de columnas si es necesario
    df.columns = [c.replace('\ufeff', '') for c in df.columns]
    
    fig = px.bar(df, x='Anno', y='Metri Quadri', color='Tipologia di Area (valore in MQ)',
                 title="Distribución de Áreas Verdes en Milán",
                 template=template, barmode='group')
    fig.show()

plot_green_evolution()

## 2. Temperaturas e Impacto Geográfico
Comparativa de temperaturas máximas y días de calor en diferentes zonas de la ciudad.

In [ ]:
def plot_temperatures():
    df = dfs['temp'].set_index('Medie Annuali').T
    df.index.name = 'Zona'
    df.reset_index(inplace=True)
    
    # Filtro por tipo de temperatura
    dropdown = widgets.Dropdown(
        options=list(df.columns[1:]),
        value=df.columns[1],
        description='Métrica:',
    )
    
    output = widgets.Output()

    def on_change(change):
        with output:
            clear_output(wait=True)
            metric = change['new']
            fig = px.bar(df, x='Zona', y=metric, color=metric,
                         color_continuous_scale='Reds',
                         title=f"Variación de {metric} por Distrito",
                         template=template)
            fig.show()

    dropdown.observe(on_change, names='value')
    display(dropdown, output)
    # Disparar evento inicial
    on_change({'new': dropdown.value})

plot_temperatures()

## 3. Techos Verdes: La Solución del Futuro
Análisis de las áreas donde se están implementando techos verdes como medida de mitigación.

In [ ]:
def plot_green_roofs():
    df = dfs['techos']
    # Agrupar por uso de suelo
    df_grouped = df.groupby('Uso_suolo_DUS')['S_copertura_Mq'].sum().reset_index()
    
    fig = px.pie(df_grouped, values='S_copertura_Mq', names='Uso_suolo_DUS',
                 title="Potencial de Techos Verdes por Uso de Suelo",
                 hole=.4, template=template)
    fig.update_traces(textinfo='percent+label')
    fig.show()

plot_green_roofs()